# Lab 3: Fully Connected Network와 CNN을 이용한 이미지 분류

이번 실습에서는 **MNIST**를 이용하여 Fully Connected Network와 CNN 기반 이미지 분류 모델을 구현합니다.
마지막에는 동일한 실습 흐름을 **Fashion-MNIST**에도 적용해 봅니다.

## 학습 목표

- 이미지 dataset을 불러오고 기본 구조를 확인합니다.
- Pixel 값을 scaling하고 label을 encoding합니다.
- Fully Connected Network를 baseline으로 구성합니다.
- Basic CNN과 Deeper CNN을 구성합니다.
- Learning curve와 test 성능을 비교합니다.
- 잘못 분류된 sample을 직접 확인합니다.
- 동일한 pipeline을 Fashion-MNIST에 재사용합니다.

> **실행 안내:** Notebook을 위에서부터 순서대로 실행하세요.  
> `X_train`, `X_test`, `model`, `history` 변수는 이후 절에서도 계속 재사용됩니다.  
> Google Colab에서 그대로 실행할 수 있습니다.

## 1. MNIST Dataset과 Preprocessing

MNIST는 손글씨 숫자 이미지 70,000개로 구성됩니다.
각 이미지는 $28 \times 28$ 크기의 grayscale image이며 총 10개의 class를 가집니다.

### 1.1 Dataset 불러오기

Keras는 MNIST dataset을 training set과 test set으로 나누어 제공합니다.

In [ ]:
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])
print(X_train.shape)
print(X_test.shape)

### 1.2 Pixel 값 확인

Grayscale image는 pixel intensity로 이루어진 2차원 행렬입니다.
다음 셀에서는 첫 번째 training image의 모든 pixel 값을 출력합니다.

In [ ]:
image = X_train[0]

for row in image:
    print(" ".join(f"{value:3d}" for value in row))

### 1.3 Input Scaling

Pixel 값은 정수 범위 $[0,255]$에서 실수 범위 $[0,1]$로 변환합니다.

이렇게 scaling하면 optimization 과정의 수치적 안정성이 좋아집니다.

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

### 1.4 Label Encoding

정수 label을 `categorical_crossentropy`에서 사용할 수 있도록
10차원 one-hot vector로 변환합니다.

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train_onehot = to_categorical(y_train, 10)
y_test_onehot = to_categorical(y_test, 10)

print("Original label:", y_train[0])
print("One-hot label:", y_train_onehot[0])

## 2. Baseline Model: Fully Connected Network

먼저 Fully Connected Network를 baseline으로 사용합니다.

이 모델은 이미지를 1차원 vector로 펼쳐 입력하기 때문에,
CNN과 달리 이미지의 명시적인 공간 구조를 사용하지 않습니다.

### 2.1 Image Flattening

각 $28 \times 28$ 이미지를 784차원 vector로 변환합니다.

$$
28 \times 28 \rightarrow 784
$$

Pixel 값 자체는 유지되지만, 2차원 공간 구조는 사라집니다.

In [ ]:
X_train = X_train.reshape(-1, 28 * 28)
X_test = X_test.reshape(-1, 28 * 28)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

### 2.2 FC Network 구성

하나의 hidden layer와 10-class softmax output layer를 사용합니다.

$$
784 \rightarrow 512 \rightarrow 10
$$

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

model = Sequential([
    Input(shape=(784,)),
    Dense(512, activation="relu"),
    Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

### 2.3 FC Network 학습

Validation split을 이용해 model selection을 수행합니다.

Early stopping은 validation loss가 가장 좋았던 epoch의 weight를 복원합니다.

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "mnist_fc_best.keras",
    monitor="val_loss",
    save_best_only=True
)

history = model.fit(
    X_train,
    y_train_onehot,
    validation_split=0.25,
    epochs=30,
    batch_size=200,
    callbacks=[early_stopping, checkpoint],
    verbose=1
)

### 2.4 FC Network 평가

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test_onehot,
    verbose=0
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

### 2.5 Learning Curve

Training loss와 validation loss를 비교하여
수렴 과정과 overfitting 여부를 확인합니다.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"],
         label="Training Loss")
plt.plot(history.history["val_loss"],
         label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

> **확인할 내용:** FC network도 MNIST를 잘 분류할 수 있지만,
> flattening 과정에서 이미지의 공간 구조가 제거됩니다.

## 3. CNN Model 1: Basic CNN

CNN은 이미지의 공간적 배치를 유지하면서
edge, stroke와 같은 local visual pattern을 직접 학습합니다.

### 3.1 Image Shape 복원

Flatten된 input을 다시 $28 \times 28 \times 1$ 형태로 변환합니다.

마지막 차원 `1`은 grayscale channel을 의미합니다.

In [ ]:
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

### 3.2 Basic CNN 구성

두 개의 convolution layer와 Batch Normalization, Max Pooling,
Fully Connected classifier를 사용합니다.

In [ ]:
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    BatchNormalization,
    ReLU,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

K.clear_session()

model = Sequential([
    Input(shape=(28, 28, 1)),

    Conv2D(32, kernel_size=(3, 3)),
    BatchNormalization(),
    ReLU(),

    Conv2D(64, kernel_size=(3, 3)),
    BatchNormalization(),
    ReLU(),

    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(10, activation="softmax")
])

model.summary()

### 3.3 Basic CNN Compile 및 Training

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "mnist_cnn1_best.keras",
    monitor="val_loss",
    save_best_only=True
)

history = model.fit(
    X_train,
    y_train_onehot,
    validation_split=0.25,
    epochs=30,
    batch_size=200,
    callbacks=[early_stopping, checkpoint],
    verbose=1
)

### 3.4 Basic CNN 평가

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test_onehot,
    verbose=0
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

### 3.5 Learning Curve

In [ ]:
import matplotlib.pyplot as plt

plt.plot(
    history.history["loss"],
    label="Training Loss"
)
plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

> **확인할 내용:** Convolution layer는 이미지의 공간 구조를 유지하면서
> local feature pattern을 직접 학습합니다.

## 4. CNN Model 2: Deeper CNN

두 번째 CNN은 feature extraction capacity를 늘리면서
spatial dimension을 단계적으로 줄입니다.

### 주요 변경 사항

- `padding="same"`을 사용하여 convolution 중 feature-map 크기를 유지합니다.
- 세 번째 convolution layer를 추가하여 더 풍부한 feature를 학습합니다.
- 두 번째 max-pooling layer로 spatial dimension을 줄입니다.
- `Flatten + Dense`를 이용해 분류에 필요한 정보를 사용합니다.
- Learning-rate scheduling을 사용하여 학습이 정체될 때 learning rate를 조정합니다.

### 4.1 Deeper CNN 구성

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    BatchNormalization,
    ReLU,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

model = Sequential([
    Input(shape=(28, 28, 1)),

    Conv2D(
        32,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False
    ),
    BatchNormalization(),
    ReLU(),

    Conv2D(
        64,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False
    ),
    BatchNormalization(),
    ReLU(),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(
        128,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False
    ),
    BatchNormalization(),
    ReLU(),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(10, activation="softmax")
])
model.summary()

### 4.2 Model Compile

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

### 4.3 Training Callbacks

원본 notebook에서는 training에서 `reduce_lr`을 사용하지만
앞선 셀에서 정의되지 않은 경우가 있었습니다.

아래 callback 설정을 실행한 뒤 학습을 진행하세요.

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "mnist_cnn2_best.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)


### 4.4 Deeper CNN 학습

In [ ]:
history = model.fit(
    X_train,
    y_train_onehot,
    validation_split=0.25,
    epochs=30,
    batch_size=128,
    callbacks=[
        early_stopping,
        checkpoint,
        reduce_lr
    ],
    verbose=1
)

### 4.5 Deeper CNN 평가

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test_onehot,
    verbose=0
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

### 4.6 Learning Curve

In [ ]:
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

> **확인할 내용:** 더 깊은 CNN은 여러 단계의 convolution을 통해
> 더 풍부한 hierarchical feature를 학습할 수 있습니다.

## 5. Error Analysis

전체 accuracy만으로는 개별적인 오분류 원인을 알기 어렵습니다.

잘못 분류된 sample을 직접 확인하면
모호한 숫자나 특이한 handwriting style을 발견할 수 있습니다.

### 5.1 잘못 분류된 Sample 확인

다음 셀에서는 test set에서 잘못 분류된 sample 중
20개를 무작위로 선택하여 실제 label과 예측 label을 함께 표시합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Predict test samples
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.argmax(y_test_onehot, axis=1)

# Find incorrect predictions
incorrect_indices = np.where(y_pred != y_true)[0]

print("Number of incorrect predictions:",
      len(incorrect_indices))

# Randomly select 20 incorrect samples
rng = np.random.default_rng(42)
sample_indices = rng.choice(
    incorrect_indices,
    size=min(20, len(incorrect_indices)),
    replace=False
)

# Visualize incorrect samples
fig, axes = plt.subplots(4, 5, figsize=(10, 8))

for ax, idx in zip(axes.ravel(), sample_indices):
    ax.imshow(X_test[idx].squeeze(), cmap="gray")
    ax.set_title(
        f"True: {y_true[idx]} | Pred: {y_pred[idx]}",
        fontsize=10
    )
    ax.axis("off")

plt.suptitle(
    "Examples of Incorrect Predictions",
    fontsize=16
)
plt.tight_layout()
plt.show()

## 6. MNIST에서 Fashion-MNIST로 확장

Fashion-MNIST는 MNIST와 동일한 데이터 형식을 가집니다.

- Training images: 60,000개
- Test images: 10,000개
- $28 \times 28$ grayscale images
- 10개의 clothing category

따라서 MNIST에서 사용한 preprocessing과 CNN pipeline을 거의 그대로 재사용할 수 있습니다.

### 6.1 Dataset 교체

Dataset import와 loading 부분만 Fashion-MNIST로 바꾸고,
나머지 preprocessing, CNN architecture, training, evaluation pipeline은 그대로 재사용합니다.

In [ ]:
from tensorflow.keras.datasets import fashion_mnist
import matplotlib.pyplot as plt

(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])
print(X_train.shape)
print(X_test.shape)

plt.imshow(X_train[0], cmap = 'gray')
plt.show()
plt.imshow(X_train[59999], cmap = 'gray')
plt.show()

### 직접 해보기

1. Fashion-MNIST를 불러오세요.
2. MNIST와 동일한 preprocessing을 적용하세요.
3. CNN architecture를 바꾸지 않고 Deeper CNN을 학습하세요.
4. MNIST와 Fashion-MNIST의 test accuracy를 비교하세요.
5. 어떤 clothing category가 자주 혼동되는지 확인하세요.

> Shirt, T-shirt, coat, pullover처럼 시각적으로 유사한 category는 구분하기 더 어려울 수 있습니다.

## 정리

- FC network는 이미지를 1차원 vector로 처리합니다.
- CNN은 이미지의 공간 구조를 유지하고 활용합니다.
- 더 깊은 convolution layer는 더 풍부한 hierarchical feature를 학습할 수 있습니다.
- Learning curve와 error analysis를 함께 사용하면 model을 더 잘 이해할 수 있습니다.
- MNIST에서 사용한 pipeline은 Fashion-MNIST에도 재사용할 수 있습니다.

TensorBoard를 이용한 training monitoring은 별도의 notebook에서 실습합니다.